# Deep Convolutional Gaussian Process for Extragalactic Globular Cluster Detection

**Based on:** Dold & Fahrion (2022), *A&A* 663, A81  
**Data:** HST/ACS imaging — F475W (*g*) + F850LP (*z*), 20 × 20 px cutouts  
**Clusters:** Virgo Cluster Catalogue (VCC) · Fornax Cluster Catalogue (FCC)

---

## Context

**Globular clusters (GCs)** are compact, gravitationally bound systems of $10^4$–$10^6$ stars, among the oldest structures in the universe ($\sim$10–13 Gyr). Their spatial distribution, kinematics, and chemical composition encode the assembly history of their host galaxy — making them one of the primary observational probes of galaxy formation.

Extragalactic GC detection is a classification problem on telescope images. A candidate source is a GC if it is spatially resolved (slightly extended relative to a point source), has the right brightness, and falls in the correct colour range. Traditional selection pipelines rely on hand-crafted cuts in magnitude–colour–concentration index space. The paper by Dold & Fahrion (2022) showed that convolutional neural networks trained on HST imaging can match or exceed expert selection, reaching **TPR ≈ 90–94 %** with **FDR ≈ 6–8 %**.

## This notebook

We replace the CNN with a **Deep Convolutional Gaussian Process (DCGP)**, gaining:

1. **Principled uncertainty quantification** — every prediction comes with a calibrated posterior variance, not just a softmax score.
2. **A richer explainability surface** — the GP's kernel structure and inducing points allow interpretation methods that have no CNN equivalent.
3. **Better handling of class imbalance** — the full dataset (~85 k sources, 1 : 3.4 GC : non-GC ratio) is used with a weighted ELBO rather than discarding majority-class examples.

The explainability work serves two goals: (i) improving the model by understanding what it learns, and (ii) giving domain experts interpretable evidence about the physical properties that distinguish GCs, supporting studies of galaxy formation.

## 1 · Setup & Imports

In [ ]:
%cd ..
!git clone https://github.com/dodo47/GCDetection

from typing import List, Tuple, Dict, Optional
import io
import traceback
import contextlib
import requests
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.distributions import Normal
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.metrics import (confusion_matrix, classification_report,
                              roc_auc_score, roc_curve)
import optuna
import seaborn as sns
from matplotlib import pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path
from utils import data

KMEANS_CACHE_DIR = Path("kmeans_cache")
KMEANS_CACHE_DIR.mkdir(exist_ok=True)
print(f"KMeans cache dir: {KMEANS_CACHE_DIR.resolve()}")

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# ── Telegram helpers ──────────────────────────────────────────────────────────

def _get_token(file_path="telegram_token.txt"):
    with open(file_path) as f:
        token, chat_id = f.read().strip().split(",")
    return token, chat_id

token, chat_id = _get_token()


def send_telegram_msg(token: str, chat_id: str, text: str) -> dict:
    """Send a plain-text Telegram message (Markdown supported)."""
    url = f"https://api.telegram.org/bot{token}/sendMessage"
    try:
        r = requests.post(url, data={"chat_id": chat_id,
                                      "text": text,
                                      "parse_mode": "Markdown"})
        return r.json()
    except Exception as e:
        print(f"Telegram sendMessage failed: {e}")


def send_telegram_photo(token: str, chat_id: str,
                        fig=None, caption: str = "") -> dict:
    """
    Send the current (or given) matplotlib figure to Telegram as a photo.

    Parameters
    ----------
    token, chat_id : Telegram credentials.
    fig            : matplotlib Figure object.  If None the current figure
                     (plt.gcf()) is used.
    caption        : Caption shown under the image (max 1024 chars).
    """
    if fig is None:
        fig = plt.gcf()

    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=120, bbox_inches="tight")
    buf.seek(0)

    url = f"https://api.telegram.org/bot{token}/sendPhoto"
    try:
        r = requests.post(url,
                          data={"chat_id": chat_id, "caption": caption[:1024]},
                          files={"photo": ("plot.png", buf, "image/png")})
        return r.json()
    except Exception as e:
        print(f"Telegram sendPhoto failed: {e}")
    finally:
        buf.close()


@contextlib.contextmanager
def notify_on_error(token: str, chat_id: str, cell_name: str):
    """
    Context manager that catches any exception, sends a Telegram alert with
    the full traceback, then re-raises so the kernel still stops.

    Usage:
        with notify_on_error(token, chat_id, "Training loop"):
            ... cell code ...
    """
    try:
        yield
    except Exception as exc:
        tb = traceback.format_exc()
        msg = (
            f"🚨 *Notebook error* in *{cell_name}*\n\n"
            f"`{type(exc).__name__}: {exc}`\n\n"
            f"```\n{tb[-1500:]}\n```"   # Telegram cap: 4096 chars total
        )
        send_telegram_msg(token, chat_id, msg)
        raise   # re-raise so the notebook stops cleanly


## 2 · Data

### Imaging

Each source is a **20 × 20 pixel cutout** in two HST/ACS bands:

| Channel | Filter | Wavelength | Physical role |
|---------|--------|------------|---------------|
| 0 | F475W (*g*) | ~475 nm | Overall brightness; traces younger/blue stellar populations |
| 1 | F850LP (*z*) | ~850 nm | Near-infrared; traces older/red populations |

The two-band setup is physically motivated: the **g − z colour** is the single most diagnostic property for GC identification. GCs display a well-known **bimodal g − z distribution** — a blue (metal-poor) peak at g − z ≈ 0.9 and a red (metal-rich) peak at g − z ≈ 1.4 — while contaminant sources (background galaxies, foreground stars, star-forming regions) scatter across a much wider colour range.

### Labels

Labels are derived from probabilistic GC membership scores ($p_\text{GC}$) computed by the original authors using multi-feature expert classification. We binarise at $p_\text{GC} \geq 0.5$.

### Class imbalance

The dataset contains ~18 k GCs and ~67 k non-GCs (ratio ≈ 1 : 3.4). The previous version of this notebook called `.sample(min_class_size)` without `replace=True`, which **downsamples** the majority class and discards ~45 k training examples. Here we keep the **full dataset** and handle the imbalance via inverse-frequency class weights in the ELBO loss (see §5).

In [ ]:
with notify_on_error(token, chat_id, "Data loading"):
    FCC, VCC = data.prepare_data()
    all_frames = pd.concat((FCC, VCC), ignore_index=True)
    all_frames = all_frames[~all_frames.frame.apply(lambda x: np.isnan(x).any())]

    counts = all_frames['y'].value_counts()
    n_neg, n_pos = int(counts[False]), int(counts[True])
    total = n_neg + n_pos
    print(f"Total samples : {total:,}")
    print(f"  GC  (pos=1) : {n_pos:,}  ({100*n_pos/total:.1f} %)")
    print(f"  non-GC (0)  : {n_neg:,}  ({100*n_neg/total:.1f} %)")

    w_pos = total / (2 * n_pos)
    w_neg = total / (2 * n_neg)
    CLASS_WEIGHTS = torch.tensor([w_neg, w_pos], dtype=torch.float32).to(device)
    print(f"\nClass weights → non-GC: {w_neg:.3f}, GC: {w_pos:.3f}")

    fig, ax = plt.subplots(figsize=(6, 3))
    sns.histplot(data=all_frames, x="y", bins=2, hue="y",
                 palette=['steelblue', 'tomato'], ax=ax)
    ax.set_title("Full dataset — class distribution")
    plt.tight_layout()
    send_telegram_photo(token, chat_id, fig,
        caption=f"✅ Data loaded\n"
                f"Total: {total:,}  |  GC: {n_pos:,} ({100*n_pos/total:.1f}%)  "
                f"|  non-GC: {n_neg:,} ({100*n_neg/total:.1f}%)")
    plt.show()


### 2.1 · Train / Val / Test Split

We use a **70 / 10 / 20** stratified random split on the merged VCC+FCC dataset for the main model.

The VCC and FCC dataframes are also kept **separate** for the cross-cluster transfer experiment (§12), where we train exclusively on one cluster and test on the other. This tests whether the model generalises across the ~3.5 Mpc distance difference between Virgo (~16.5 Mpc) and Fornax (~20 Mpc), which slightly changes angular sizes and apparent magnitudes of GCs.

In [ ]:
with notify_on_error(token, chat_id, "Data splits"):
    def make_splits(df, train_frac=0.70, val_frac=0.10, seed=42):
        df = df.sample(frac=1, random_state=seed).reset_index(drop=True)
        n = len(df)
        n_train = int(n * train_frac)
        n_val   = int(n * val_frac)
        return df[:n_train], df[n_train:n_train+n_val], df[n_train+n_val:]

    train_df, val_df, test_df = make_splits(all_frames)
    print(f"Train: {len(train_df):,}  |  Val: {len(val_df):,}  |  Test: {len(test_df):,}")

    fcc_df = FCC[~FCC.frame.apply(lambda x: np.isnan(x).any())]
    vcc_df = VCC[~VCC.frame.apply(lambda x: np.isnan(x).any())]


## 3 · Dataset & DataLoaders

In [ ]:
with notify_on_error(token, chat_id, "Dataset & DataLoaders"):
    class FrameDataset(Dataset):
        """Two-channel (g, z) image dataset for GC classification."""
    
        def __init__(self, df: pd.DataFrame, transform=None):
            self.frames = np.stack(df.frame.values).astype('float32')  # (N, 2, 20, 20)
            self.labels = np.array(df.y, dtype=np.int64)
            self.transform = transform
    
        def __len__(self):
            return len(self.labels)
    
        def __getitem__(self, idx):
            img = self.frames[idx]
            if self.transform:
                ch0 = self.transform(img[0])
                ch1 = self.transform(img[1])
                img = np.stack([ch0, ch1])
            return torch.tensor(img).float(), torch.tensor(self.labels[idx])
    
    
    def make_loader(df, batch_size=64, shuffle=True, num_workers=4):
        return DataLoader(FrameDataset(df), batch_size=batch_size,
                          shuffle=shuffle, num_workers=num_workers, pin_memory=True)
    
    train_dl = make_loader(train_df, shuffle=True)
    val_dl   = make_loader(val_df,   shuffle=False)
    test_dl  = make_loader(test_df,  shuffle=False)


## 4 · Deep Convolutional Gaussian Process

### 4.1 Gaussian Processes — brief recap

A **Gaussian Process** is a distribution over functions $f \sim \mathcal{GP}(m, k)$, fully specified by a mean function $m(\mathbf{x})$ and a **covariance (kernel) function** $k(\mathbf{x}, \mathbf{x}')$. For any finite collection of inputs $\mathbf{X}$, the function values $\mathbf{f} = f(\mathbf{X})$ follow a multivariate Gaussian:

$$p(\mathbf{f} \mid \mathbf{X}) = \mathcal{N}(\mathbf{f} \mid \mathbf{m}(\mathbf{X}),\; K_{\mathbf{X}\mathbf{X}})$$

where $[K_{\mathbf{X}\mathbf{X}}]_{ij} = k(\mathbf{x}_i, \mathbf{x}_j)$. We use the **RBF (squared-exponential) kernel**:

$$k(\mathbf{x}, \mathbf{x}') = \sigma^2 \exp\!\left(-\frac{\|\mathbf{x} - \mathbf{x}'\|^2}{2\ell^2}\right)$$

with learnable signal variance $\sigma^2$ and lengthscale $\ell$, both parameterised in log-space to ensure positivity.

### 4.2 Sparse variational inference and inducing points

A GP conditioned on $N$ training points requires $O(N^3)$ computation. For our ~60 k training images this is intractable. We use the **sparse variational GP** framework (Titsias 2009; Hensman et al. 2013), which introduces $M \ll N$ **inducing inputs** $\mathbf{Z} \in \mathbb{R}^{M \times P}$ with corresponding inducing outputs $\mathbf{u} = f(\mathbf{Z})$.

The true posterior $p(\mathbf{f} \mid \mathbf{y})$ is approximated by a variational distribution:

$$q(\mathbf{f}, \mathbf{u}) = p(\mathbf{f} \mid \mathbf{u})\, q(\mathbf{u}), \qquad q(\mathbf{u}) = \mathcal{N}(\mathbf{u} \mid \mathbf{m}, \mathbf{S})$$

where $\mathbf{m} \in \mathbb{R}^{M \times C}$ and $\mathbf{S} = \mathbf{L}_S \mathbf{L}_S^\top \in \mathbb{R}^{M \times M}$ (full-rank, PD by construction) are variational parameters learned by maximising the ELBO. The predictive distribution at a new point $\mathbf{x}^*$ is:

$$q(f^*) = \mathcal{N}\!\left(f^* \;\middle|\; \mathbf{A}\mathbf{m},\; k_{**} - \mathbf{A}(K_{\mathbf{Z}\mathbf{Z}} - \mathbf{S})\mathbf{A}^\top\right)$$

where $\mathbf{A} = K_{\mathbf{x}^*\mathbf{Z}}\,K_{\mathbf{Z}\mathbf{Z}}^{-1}$. This costs $O(NM^2)$ — tractable for our scale.

The inducing points $\mathbf{Z}$ are **initialised via k-means on training patches** (following the paper) and then treated as learnable parameters. The k-means results are cached to disk so repeated Optuna trials with the same $(M, k, C_\text{in})$ config skip the expensive fit entirely.

### 4.3 Convolutional GP layers

A standard GP operates on vectors. The **Convolutional GP** (van der Wilk et al. 2017) extends this to images by applying the GP to **image patches** rather than full images.

For an input $\mathbf{X} \in \mathbb{R}^{B \times C \times H \times W}$, we extract all $k \times k$ patches to form $\mathbf{P} \in \mathbb{R}^{B \times N_p \times P}$ where $P = C \cdot k^2$ and $N_p$ is the number of spatial positions. The GP is applied independently to each patch, and outputs are arranged back into a spatial feature map.

### 4.4 Two-layer architecture

```
Input (B, 2, 20, 20)
    │
    ▼
ExtractorConvGP   — in_channels=2, out_channels=C, kernel=k, M inducing pts
    │  Applies sparse GP patch-wise → samples a spatial feature map
    │  Output: (B, C, H', W')
    ▼
ClassifierConvGP  — in_channels=C, out_channels=2 (classes), kernel=k, M inducing pts
    │  Applies sparse GP patch-wise → averages over spatial positions
    │  Output: Gaussian (mean, var) over class logits, shape (B, 2)
    ▼
Predictive distribution  q(f*) = N(mean, var)
    │  Monte-Carlo softmax → P(GC)
    ▼
Binary classification
```

The **extractor** produces a sampled feature map via the reparameterisation trick. The **classifier** aggregates spatial information into a class-logit Gaussian, whose mean and variance are both propagated to the loss. This end-to-end uncertainty quantification is the key difference from a CNN.

In [ ]:
with notify_on_error(token, chat_id, 'ConvGPBase definition'):
    class ConvGPBase(nn.Module):
        """Shared base for all ConvGP layers."""
    
        def __init__(self, device, in_channels: int, out_channels: int,
                     kernel_size: int, M: int, padding=0, stride=1,
                     s: float = 0.5413, ls: float = 0.5413, name: str = None):
            super().__init__()
            self.k, self.out_channels, self.M = kernel_size, out_channels, M
            self.P = in_channels * kernel_size * kernel_size
            self.padding, self.stride = padding, stride
            self.device = device
            self.name = name
    
            # Inducing inputs  (M, P)
            self.Z = nn.Parameter(torch.randn(M, self.P) * 0.1)
    
            # RBF hyper-parameters (log-space for positivity)
            self.log_s  = nn.Parameter(torch.tensor(float(np.log(s))))
            self.log_ls = nn.Parameter(torch.tensor(float(np.log(ls))))
    
        @property
        def sigma(self):
            return F.softplus(self.log_s)
    
        @property
        def lengthscale(self):
            return F.softplus(self.log_ls)
    
        def _rbf(self, x1: torch.Tensor, x2: torch.Tensor) -> torch.Tensor:
            """RBF(x1, x2) with shape broadcasting."""
            s, ls = self.sigma, self.lengthscale
            if x1.dim() == 2 and x2.dim() == 2:
                dist_sq = torch.cdist(x1, x2) ** 2
            else:
                # batched: (B, n, P) vs (B, M, P) or (1, M, P)
                if x2.dim() == 2:
                    x2 = x2.unsqueeze(0)
                dist_sq = torch.cdist(x1, x2) ** 2
            return (s ** 2) * torch.exp(-dist_sq / (2 * ls ** 2))
    
        def _chol_kzz(self) -> torch.Tensor:
            K = self._rbf(self.Z, self.Z) + 1e-5 * torch.eye(self.M, device=self.device)
            return torch.linalg.cholesky(K)
    
        def initialize_inducing_kmeans(self, patches: torch.Tensor,
                                       cache_dir: Path = None):
            """
            Fit KMeans on `patches` and store the cluster centers in self.Z.

            Results are cached to disk keyed by (M, P, layer_name) so that
            every Optuna trial reusing the same (M, kernel_size, in_channels)
            combination skips the expensive fit entirely.

            Cache filename:
                <cache_dir>/kmeans_M{M}_P{P}_{name}.npy   — float32 (M, P)

            To force a refit, simply delete the relevant .npy file.
            """
            if cache_dir is None:
                cache_dir = KMEANS_CACHE_DIR

            cache_path = cache_dir / f"kmeans_M{self.M}_P{self.P}_{self.name}.npy"

            if cache_path.exists():
                # ── cache hit ────────────────────────────────────────────────
                centers = np.load(str(cache_path))          # (M, P) float32
                self.Z.data.copy_(torch.from_numpy(centers).to(self.device))
                print(f"  [KMeans cache] HIT  — {cache_path.name}")
                return

            # ── cache miss: run KMeans then persist ──────────────────────────
            print(f"  [KMeans cache] MISS — computing {cache_path.name} ...")
            patches_np = patches.detach().cpu().numpy()
            n_samples  = min(patches_np.shape[0], self.M * 10)

            if n_samples < self.M:
                idx     = np.random.choice(patches_np.shape[0], self.M, replace=True)
                centers = patches_np[idx]
            else:
                idx = np.random.choice(patches_np.shape[0], n_samples, replace=False)
                km  = KMeans(n_clusters=self.M, random_state=42, n_init=1)
                km.fit(patches_np[idx])
                centers = km.cluster_centers_               # (M, P) float64

            centers = centers.astype(np.float32)
            np.save(str(cache_path), centers)
            print(f"  [KMeans cache] saved  {cache_path.name}")

            self.Z.data.copy_(torch.from_numpy(centers).to(self.device))


In [ ]:
with notify_on_error(token, chat_id, 'ExtractorConvGP definition'):
    class ExtractorConvGP(ConvGPBase):
        """GP layer that produces a sampled spatial feature map."""
    
        def __init__(self, device, in_channels, out_channels, kernel_size, M,
                     padding=0, stride=1, s=0.5413, ls=0.5413, name=None):
            super().__init__(device, in_channels, out_channels, kernel_size,
                             M, padding, stride, s, ls, name)
            # Variational mean (M, out_channels)
            self.m   = nn.Parameter(torch.randn(M, out_channels) * 0.01)
            # Lower-triangular Cholesky of variational covariance (M, M)
            self.L_S = nn.Parameter(torch.eye(M) * 0.1)
    
        @property
        def S(self) -> torch.Tensor:
            L = torch.tril(self.L_S)
            return L @ L.T + 1e-5 * torch.eye(self.M, device=self.device)
    
        def kl_divergence(self) -> torch.Tensor:
            L_z = self._chol_kzz()
            S = self.S
            K_inv_m = torch.cholesky_solve(self.m, L_z)           # (M, out_ch)
            K_inv_S = torch.cholesky_solve(S, L_z)                 # (M, M)
            L_S_chol = torch.linalg.cholesky(S + 1e-6*torch.eye(self.M, device=self.device))
            kl = 0.5 * (
                torch.trace(K_inv_S) * self.out_channels
                + (self.m.T @ K_inv_m).trace()
                - self.M * self.out_channels
                + 2 * self.out_channels * L_z.diagonal().log().sum()
                - 2 * self.out_channels * L_S_chol.diagonal().log().sum()
            )
            return kl.clamp(min=0.0)
    
        def forward(self, x: torch.Tensor) -> torch.Tensor:
            B = x.shape[0]
            # Extract patches  →  (B, n_patches, P)
            patches = F.unfold(x, kernel_size=self.k,
                               padding=self.padding, stride=self.stride)
            patches = patches.transpose(1, 2)                          # (B, np, P)
            n_patches = patches.shape[1]
    
            L_z = self._chol_kzz()                                     # (M, M)
            Z_exp = self.Z.unsqueeze(0).expand(B, -1, -1)              # (B, M, P)
            K_xz = self._rbf(patches, Z_exp)                           # (B, np, M)
    
            A = torch.cholesky_solve(K_xz.transpose(-1, -2), L_z
                                     ).transpose(-1, -2)               # (B, np, M)
            mean = A @ self.m                                          # (B, np, out_ch)
    
            # Diagonal posterior variance per patch
            S = self.S
            var_diag = (A @ S * A).sum(-1).clamp(min=1e-6)            # (B, np)
            eps = torch.randn_like(mean)
            sample = mean + var_diag.sqrt().unsqueeze(-1) * eps        # (B, np, out_ch)
    
            # Reshape to spatial feature map
            h_out = (x.shape[2] + 2*self.padding - self.k) // self.stride + 1
            w_out = (x.shape[3] + 2*self.padding - self.k) // self.stride + 1
            return sample.transpose(1, 2).view(B, self.out_channels, h_out, w_out)


In [ ]:
with notify_on_error(token, chat_id, 'ClassifierConvGP definition'):
    class ClassifierConvGP(ConvGPBase):
        """GP layer that aggregates spatial features into class-logit Gaussians."""
    
        def __init__(self, device, in_channels, in_shape: Tuple[int,int],
                     out_channels, kernel_size, M,
                     padding=0, stride=1, s=0.5413, ls=0.5413, name=None):
            super().__init__(device, in_channels, out_channels, kernel_size,
                             M, padding, stride, s, ls, name)
            h_out = (in_shape[0] + 2*padding - kernel_size) // stride + 1
            w_out = (in_shape[1] + 2*padding - kernel_size) // stride + 1
            self.n_patches = h_out * w_out
    
            self.m   = nn.Parameter(torch.randn(M, out_channels) * 0.01)
            self.L_S = nn.Parameter(torch.eye(M) * 0.1)
    
        @property
        def S(self) -> torch.Tensor:
            L = torch.tril(self.L_S)
            return L @ L.T + 1e-5 * torch.eye(self.M, device=self.device)
    
        def kl_divergence(self) -> torch.Tensor:
            L_z = self._chol_kzz()
            S = self.S
            K_inv_m = torch.cholesky_solve(self.m, L_z)
            K_inv_S = torch.cholesky_solve(S, L_z)
            L_S_chol = torch.linalg.cholesky(S + 1e-6*torch.eye(self.M, device=self.device))
            kl = 0.5 * (
                torch.trace(K_inv_S) * self.out_channels
                + (self.m.T @ K_inv_m).trace()
                - self.M * self.out_channels
                + 2 * self.out_channels * L_z.diagonal().log().sum()
                - 2 * self.out_channels * L_S_chol.diagonal().log().sum()
            )
            return kl.clamp(min=0.0)
    
        def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
            B = x.shape[0]
            patches = F.unfold(x, kernel_size=self.k,
                               padding=self.padding, stride=self.stride)
            patches = patches.transpose(1, 2)                          # (B, np, P)
    
            L_z = self._chol_kzz()
            Z_exp = self.Z.unsqueeze(0).expand(B, -1, -1)
            K_xz  = self._rbf(patches, Z_exp)                         # (B, np, M)
    
            A = torch.cholesky_solve(K_xz.transpose(-1, -2), L_z
                                     ).transpose(-1, -2)               # (B, np, M)
    
            # Mean prediction: average over patches  →  (B, out_channels)
            mean = (A @ self.m).mean(dim=1)
    
            # Diagonal variance, averaged over patches  →  (B, out_channels)
            S = self.S
            var = (A @ S * A).sum(-1).mean(dim=1)                      # (B,)
            var = var.unsqueeze(-1).expand(-1, self.out_channels)      # (B, out_ch)
    
            return mean, var


In [ ]:
with notify_on_error(token, chat_id, 'DeepCGP definition'):
    class DeepCGP(nn.Module):
        """Two-layer Deep Convolutional GP."""
    
        def __init__(self, layers: List[nn.Module]):
            super().__init__()
            self.layers = nn.ModuleList(layers)
            self.device = layers[0].device
    
        def forward(self, x: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
            for layer in self.layers:
                if isinstance(layer, ClassifierConvGP):
                    return layer(x)
                x = layer(x)
            raise RuntimeError("No ClassifierConvGP found in layers.")
    
        def kl_divergence(self) -> torch.Tensor:
            return sum(l.kl_divergence() for l in self.layers)
    
        @torch.no_grad()
        def initialize_inducing_points(self, dataloader: DataLoader):
            self.eval()
            x, _ = next(iter(dataloader))
            x = x.to(self.device)
            current = x
            for layer in self.layers:
                patches = F.unfold(current, kernel_size=layer.k,
                                   padding=layer.padding, stride=layer.stride)
                patches = patches.transpose(1, 2).reshape(-1, layer.P)
                layer.initialize_inducing_kmeans(patches)
                if isinstance(layer, ClassifierConvGP):
                    break
                # Deterministic mean pass for next layer
                s, ls = layer.sigma, layer.lengthscale
                patches_r = patches.view(x.shape[0], -1, layer.P)
                K_zz = layer._rbf(layer.Z, layer.Z) + 1e-5*torch.eye(layer.M, device=self.device)
                K_xz = layer._rbf(patches_r, layer.Z.unsqueeze(0).expand(x.shape[0], -1, -1))
                L_z  = torch.linalg.cholesky(K_zz)
                A    = torch.cholesky_solve(K_xz.transpose(-1,-2), L_z).transpose(-1,-2)
                mean = A @ layer.m
                h_out = (current.shape[2]+2*layer.padding-layer.k)//layer.stride+1
                w_out = (current.shape[3]+2*layer.padding-layer.k)//layer.stride+1
                current = mean.transpose(1,2).view(x.shape[0], layer.out_channels, h_out, w_out)


## 5 · Training Objective: Weighted ELBO

### 5.1 The Evidence Lower Bound

Variational inference maximises the **ELBO** (Evidence Lower BOund), which is a lower bound on the log-marginal likelihood $\log p(\mathbf{y})$:

$$\mathcal{L} = \underbrace{\mathbb{E}_{q(\mathbf{f})}\!\left[\log p(\mathbf{y} \mid \mathbf{f})\right]}_{\text{data fit}} - \underbrace{\mathrm{KL}\!\left[q(\mathbf{u}) \;\|\; p(\mathbf{u})\right]}_{\text{regularisation}}$$

The **KL term** penalises the variational posterior $q(\mathbf{u})$ for deviating from the GP prior $p(\mathbf{u}) = \mathcal{N}(\mathbf{0}, K_{\mathbf{Z}\mathbf{Z}})$. For Gaussian distributions it has a closed form:

$$\mathrm{KL}[q(\mathbf{u}) \| p(\mathbf{u})] = \frac{1}{2}\left[\mathrm{tr}(K_{\mathbf{ZZ}}^{-1}\mathbf{S}) + \mathbf{m}^\top K_{\mathbf{ZZ}}^{-1}\mathbf{m} - M + \log\frac{|K_{\mathbf{ZZ}}|}{|\mathbf{S}|}\right]$$

The **data-fit term** involves an expectation over $q(\mathbf{f})$ that is intractable for the softmax likelihood. We estimate it via **doubly stochastic MC sampling** (Salimbeni & Deisenroth 2017): at each training step we sample $S$ function values from $q(f^*)$ using the reparameterisation trick and average the log-likelihoods.

### 5.2 KL warm-up

The KL term is annealed with a scalar $\beta$:

$$\mathcal{L}_\beta = \mathbb{E}_{q}[\log p(\mathbf{y}\mid\mathbf{f})] - \beta \cdot \mathrm{KL}[q\|p]$$

$\beta$ is linearly increased from 0 to 1 over the first `warmup_epochs` epochs. Starting at $\beta = 0$ (pure data-fit) prevents the prior from collapsing the variational posterior before it has learned anything useful from data.

### 5.3 Class-weighted data-fit term

For a 1 : 3.4 imbalanced dataset, unweighted cross-entropy causes the model to over-predict the majority class. We assign inverse-frequency weights:

$$w_c = \frac{N}{2 \cdot N_c}$$

so that $w_\text{GC} \approx 2.2$ and $w_\text{non-GC} \approx 0.65$. These are passed to `F.cross_entropy(..., weight=class_weights)` inside the MC loop, effectively up-weighting GC misclassifications in the data-fit term without discarding any data.

In [ ]:
with notify_on_error(token, chat_id, "ELBO loss definition"):
    def elbo_loss(model: DeepCGP,
                  x: torch.Tensor,
                  y: torch.Tensor,
                  class_weights: torch.Tensor,
                  num_samples: int = 10,
                  beta: float = 1.0,
                  dataset_size: int = 1000) -> Tuple[torch.Tensor, dict]:
        """
        Compute the weighted ELBO.
    
        Monte-Carlo estimate with `num_samples` samples from q(f*) via
        the reparameterisation trick baked into the forward pass.
        """
        # Average NLL over MC samples
        nll_sum = torch.zeros(1, device=x.device)
        for _ in range(num_samples):
            mean, var = model(x)
            std = var.clamp(min=1e-6).sqrt()
            # Reparameterised logits
            eps = torch.randn_like(mean)
            logits = mean + std * eps                              # (B, n_classes)
            nll_sum = nll_sum + F.cross_entropy(logits, y,
                                                weight=class_weights)
        nll = nll_sum / num_samples
    
        kl   = model.kl_divergence() / dataset_size
        loss = nll + beta * kl
    
        with torch.no_grad():
            mean, _ = model(x)
            preds = mean.argmax(dim=1)
            acc   = (preds == y).float().mean().item()
    
        return loss, {'loss': loss.item(), 'nll': nll.item(),
                      'kl': kl.item(),  'accuracy': acc}


## 6 · Training & Validation

Training uses **Adam** with gradient clipping (max norm 5.0) to guard against occasional large gradients from the Cholesky solves. The learning rate is decayed with `CosineAnnealingLR` for the final model.

Each validation pass sets $\beta = 0$ (KL term excluded) so the reported validation loss is a pure data-fit metric, comparable across epochs regardless of the current warm-up schedule.

In [ ]:
with notify_on_error(token, chat_id, "Train/val loop definitions"):
    def train_epoch(model, loader, optimizer, epoch,
                    class_weights, num_samples=10,
                    beta_max=1.0, warmup_epochs=20,
                    dataset_size=1000):
        model.train()
        beta = beta_max * min(1.0, (epoch + 1) / max(1, warmup_epochs))
        totals = {k: 0.0 for k in ('loss','nll','kl','accuracy')}
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            loss, m = elbo_loss(model, x, y, class_weights,
                                num_samples, beta, dataset_size)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 5.0)
            optimizer.step()
            for k in totals: totals[k] += m[k]
        n = len(loader)
        return {k: v/n for k, v in totals.items()}
    
    
    @torch.no_grad()
    def validate(model, loader, class_weights, num_samples=10, dataset_size=1):
        model.eval()
        totals = {k: 0.0 for k in ('loss','nll','accuracy')}
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            _, m = elbo_loss(model, x, y, class_weights,
                             num_samples, beta=0.0,
                             dataset_size=dataset_size)
            for k in totals: totals[k] += m[k]
        n = len(loader)
        return {k: v/n for k, v in totals.items()}


## 7 · Hyperparameter Search with Optuna

We optimise six hyperparameters with **TPE (Tree-structured Parzen Estimator)** sampling:

| Parameter | Search space | Effect |
|-----------|-------------|--------|
| `M` | {32, 64, 128, 256} | Number of inducing points — controls expressivity vs. $O(M^2)$ cost |
| `kernel_size` | {3, 5} | Spatial context per patch; 5 covers ~5 px radius |
| `out_channels` | {8, 16, 32} | Feature dimensionality between GP layers |
| `log_s` | [−1, 1] | Log signal variance — controls kernel amplitude |
| `log_ls` | [−1, 1] | Log lengthscale — controls how quickly similarity decays |
| `learning_rate` | [1e-4, 1e-2] (log) | Adam step size |

**Pruning** uses `MedianPruner(n_startup_trials=5, n_warmup_steps=3)`: after the first 5 complete trials, any new trial whose validation loss at epoch $e \geq 3$ is above the median of all completed trials at that step is stopped early. Crucially, `trial.report(val_loss, step=epoch)` is called inside the training loop every epoch — without this the pruner has no intermediate values and never fires.

**Numerical robustness**: Cholesky decompositions can fail (non-positive-definite kernel matrix) for extreme values of `log_s` / `log_ls`. These trials are caught and returned as a large finite penalty (`1e4`) rather than `inf` (which breaks TPE) or a raised exception (which marks the trial as FAILED and excludes it from the pruner's reference distribution).

In [ ]:
with notify_on_error(token, chat_id, "Optuna study definition"):
    def build_model(params, device, input_shape=(20,20), in_channels=2, n_classes=2):
        M, ks, out_ch = params['M'], params['kernel_size'], params['out_channels']
        s   = np.exp(params['log_s'])
        ls  = np.exp(params['log_ls'])
        pad = ks // 2
        h_out = (input_shape[0] + 2*pad - ks) + 1
        w_out = (input_shape[1] + 2*pad - ks) + 1
        extractor  = ExtractorConvGP(device, in_channels, out_ch, ks, M,
                                      pad, 1, s, ls, 'extractor')
        classifier = ClassifierConvGP(device, out_ch, (h_out, w_out),
                                       n_classes, ks, M, pad, 1, s, ls, 'classifier')
        model = DeepCGP([extractor, classifier]).to(device)
        model.learning_rate = params.get('learning_rate', 1e-3)
        return model

    def optuna_objective(trial, train_df, val_df, n_epochs=10, batch_size=64):
        params = {
            'M'            : trial.suggest_categorical('M', [32, 64, 128, 256]),
            'kernel_size'  : trial.suggest_categorical('kernel_size', [3, 5]),
            'out_channels' : trial.suggest_categorical('out_channels', [8, 16, 32]),
            'log_s'        : trial.suggest_float('log_s',  -1.0, 1.0),
            'log_ls'       : trial.suggest_float('log_ls', -1.0, 1.0),
            'learning_rate': trial.suggest_float('learning_rate', 1e-4, 1e-2, log=True),
        }
        tr_dl = make_loader(train_df, batch_size, shuffle=True,  num_workers=0)
        v_dl  = make_loader(val_df,   batch_size, shuffle=False, num_workers=0)
        try:
            model = build_model(params, device)
            model.initialize_inducing_points(tr_dl)
            opt   = torch.optim.Adam(model.parameters(), lr=params['learning_rate'])
            sched = torch.optim.lr_scheduler.StepLR(opt,
                        step_size=max(1, n_epochs//3), gamma=0.5)
            m = {}
            for epoch in range(n_epochs):
                train_epoch(model, tr_dl, opt, epoch, CLASS_WEIGHTS,
                            dataset_size=len(train_df))
                sched.step()

                # Validate every epoch and report to pruner — without this
                # trial.report() call, should_prune() has no data and never fires.
                m = validate(model, v_dl, CLASS_WEIGHTS, dataset_size=len(val_df))
                trial.report(m['loss'], step=epoch)

                if trial.should_prune():
                    raise optuna.exceptions.TrialPruned()
            send_telegram_msg(token, chat_id,
            f"*Optuna trial {trial.number}*\n"
            f"loss: `{m['loss']:.4f}`")
            return m['loss']
        except optuna.exceptions.TrialPruned:
            raise
        except:
            raise optuna.exceptions.TrialPruned()

    def run_study(train_df, val_df, n_trials=20, n_epochs=10):
        study = optuna.create_study(
            direction='minimize',
            pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=3))
        study.optimize(
            lambda t: optuna_objective(t, train_df, val_df, n_epochs),
            n_trials=n_trials, show_progress_bar=True)
        print("Best params:", study.best_params)
        print(f"Best val loss: {study.best_value:.4f}")
        return study



## 8 · Final Model Training

In [ ]:
with notify_on_error(token, chat_id, "Optuna search + final training"):
    study      = run_study(train_df, val_df, n_trials=20, n_epochs=10)
    best_params = study.best_params

    send_telegram_msg(token, chat_id,
        f"✅ *Optuna done*\n"
        f"Best val loss: `{study.best_value:.4f}`\n"
        f"Params: `{best_params}`")

    best_model = build_model(best_params, device)
    best_model.initialize_inducing_points(train_dl)

    optimizer = torch.optim.Adam(best_model.parameters(),
                                  lr=best_params['learning_rate'])
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)

    EPOCHS  = 50
    history = {'train_loss':[], 'train_acc':[], 'val_loss':[], 'val_acc':[]}

    for epoch in range(EPOCHS):
        tr = train_epoch(best_model, train_dl, optimizer, epoch,
                         CLASS_WEIGHTS, warmup_epochs=10,
                         dataset_size=len(train_df))
        vl = validate(best_model, val_dl, CLASS_WEIGHTS,
                      dataset_size=len(val_df))
        scheduler.step()

        history['train_loss'].append(tr['loss'])
        history['train_acc'].append(tr['accuracy'])
        history['val_loss'].append(vl['loss'])
        history['val_acc'].append(vl['accuracy'])

        if (epoch+1) % 10 == 0:
            print(f"Epoch {epoch+1:3d}  "
                  f"train loss {tr['loss']:.4f}  acc {tr['accuracy']:.3f}  |  "
                  f"val loss {vl['loss']:.4f}  acc {vl['accuracy']:.3f}")
            send_telegram_msg(token, chat_id,
                f"🔄 *Training* — epoch {epoch+1}/{EPOCHS}\n"
                f"train loss `{tr['loss']:.4f}` acc `{tr['accuracy']:.3f}`\n"
                f"val   loss `{vl['loss']:.4f}` acc `{vl['accuracy']:.3f}`")

    # ── learning curves ───────────────────────────────────────────────────────
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history['train_loss'], label='Train')
    axes[0].plot(history['val_loss'],   label='Val')
    axes[0].set_title('Loss'); axes[0].legend(); axes[0].set_xlabel('Epoch')
    axes[1].plot(history['train_acc'], label='Train')
    axes[1].plot(history['val_acc'],   label='Val')
    axes[1].set_title('Accuracy'); axes[1].legend(); axes[1].set_xlabel('Epoch')
    plt.tight_layout()
    send_telegram_photo(token, chat_id, fig,
        caption=f"✅ Training complete ({EPOCHS} epochs)\n"
                f"Final val loss: {history['val_loss'][-1]:.4f}  "
                f"acc: {history['val_acc'][-1]:.3f}")
    plt.show()


In [ ]:
with notify_on_error(token, chat_id, "Model save/load"):
    torch.save({'model_state_dict': best_model.state_dict(),
                'hyperparams': best_params}, "best_model.pth")
    print("Model saved.")

    def load_model(path, device):
        ckpt = torch.load(path, map_location=device, weights_only=False)
        model = build_model(ckpt['hyperparams'], device)
        model.load_state_dict(ckpt['model_state_dict'])
        model.eval()
        return model

    best_model = load_model("best_model.pth", device)
    send_telegram_msg(token, chat_id, "✅ Model saved and reloaded from disk.")


## 9 · Test-Set Evaluation

We report the same metrics as Table 3 and Table 5 of the paper, enabling direct comparison:

- **TPR** (True Positive Rate / recall): fraction of real GCs correctly detected — the primary science metric, since missed GCs are lost science
- **FDR** (False Discovery Rate): fraction of detected sources that are not GCs — drives contamination in the final catalogue
- **AUC-ROC**: threshold-independent ranking performance

Predictions are made by drawing $S = 500$ Monte-Carlo samples from $q(f^*) = \mathcal{N}(\text{mean}, \text{var})$, applying softmax to each, and averaging to obtain $\hat{p}(\text{GC})$. This fully marginalises over the epistemic uncertainty in the function value at test time, which is a more principled estimate than simply taking `softmax(mean)`.

In [ ]:
with notify_on_error(token, chat_id, "Test-set evaluation"):
    def predict_dataset(model, loader, n_mc=500):
        model.eval()
        all_labels, all_probs, all_stds = [], [], []
        with torch.no_grad():
            for x, y in loader:
                x = x.to(device)
                mean, var = model(x)
                std = var.clamp(min=1e-6).sqrt()
                dist = Normal(mean, std)
                samples = dist.rsample((n_mc,))
                probs = F.softmax(samples, dim=-1).mean(0)
                all_probs.append(probs.cpu().numpy())
                all_stds.append(std.cpu().numpy())
                all_labels.append(y.numpy())
        labels = np.concatenate(all_labels)
        probs  = np.concatenate(all_probs,  axis=0)
        stds   = np.concatenate(all_stds,   axis=0)
        preds  = (probs[:, 1] >= 0.5).astype(int)
        return labels, probs, preds, stds

    y_true, y_proba, y_pred, y_std = predict_dataset(best_model, test_dl)

    report = classification_report(y_true, y_pred, target_names=['non-GC','GC'])
    print("\n── Classification Report ──────────────────────────────")
    print(report)

    cm  = confusion_matrix(y_true, y_pred)
    tn, fp, fn, tp = cm.ravel()
    tpr = tp / (tp + fn)
    fdr = fp / (fp + tp)
    auc = roc_auc_score(y_true, y_proba[:,1])
    print(f"TPR: {tpr:.3f}  FDR: {fdr:.3f}  AUC: {auc:.3f}")

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['non-GC','GC'], yticklabels=['non-GC','GC'],
                ax=axes[0])
    axes[0].set_ylabel('True'); axes[0].set_title('Confusion Matrix')

    fpr_arr, tpr_arr, _ = roc_curve(y_true, y_proba[:,1])
    axes[1].plot(fpr_arr, tpr_arr, lw=2, label=f'AUC={auc:.3f}')
    axes[1].plot([0,1],[0,1],'--', color='grey')
    axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR')
    axes[1].set_title('ROC Curve'); axes[1].legend()
    plt.tight_layout()

    send_telegram_photo(token, chat_id, fig,
        caption=f"✅ Test evaluation\n"
                f"TPR: {tpr:.3f}  |  FDR: {fdr:.3f}  |  AUC: {auc:.3f}")
    plt.show()


## 10 · Science Validation — g−z Colour Distribution Recovery

A model that genuinely identifies GCs by their physical properties — not by spurious correlates — should recover the known **bimodal g−z colour distribution** in its detected catalogue.

GC colour bimodality is one of the most robustly observed properties of GC systems around massive galaxies. The two peaks are interpreted as:

- **Blue peak** (g − z ≈ 0.9): metal-poor GCs, typically found in the outer halo, believed to have formed in low-mass progenitors before assembly
- **Red peak** (g − z ≈ 1.4): metal-rich GCs, more centrally concentrated, believed to have formed in-situ during the main star-forming epoch of the host galaxy

We compare the colour distributions of: (i) ground-truth GCs, (ii) model-detected GCs, (iii) false positives, and (iv) missed GCs. If the model is working correctly, (ii) should mirror (i). The colour distributions of (iii) and (iv) reveal which part of colour space the model struggles with — for example, if false positives cluster at intermediate colours, the model may be confused by reddened foreground stars or compact background galaxies.

In [ ]:
with notify_on_error(token, chat_id, "g-z colour distribution recovery"):
    colour_col   = 'colour'
    test_colours = test_df[colour_col].values

    fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=False)

    gc_mask  = y_true == 1
    ngc_mask = y_true == 0
    axes[0].hist(test_colours[gc_mask],  bins=40, color='tomato',    alpha=0.7,
                 label='True GC',     density=True)
    axes[0].hist(test_colours[ngc_mask], bins=40, color='steelblue', alpha=0.5,
                 label='True non-GC', density=True)
    axes[0].set_title('Ground Truth'); axes[0].set_xlabel('g–z colour'); axes[0].legend()

    det_gc  = y_pred == 1
    det_ngc = y_pred == 0
    axes[1].hist(test_colours[det_gc],  bins=40, color='tomato',    alpha=0.7,
                 label='Detected GC',     density=True)
    axes[1].hist(test_colours[det_ngc], bins=40, color='steelblue', alpha=0.5,
                 label='Detected non-GC', density=True)
    axes[1].set_title('Model Detections'); axes[1].set_xlabel('g–z colour'); axes[1].legend()

    fp_mask = (y_pred == 1) & (y_true == 0)
    fn_mask = (y_pred == 0) & (y_true == 1)
    axes[2].hist(test_colours[fp_mask], bins=40, color='orange', alpha=0.7,
                 label=f'FP ({fp_mask.sum()})',          density=True)
    axes[2].hist(test_colours[fn_mask], bins=40, color='purple', alpha=0.7,
                 label=f'FN / missed ({fn_mask.sum()})', density=True)
    axes[2].set_title('Errors: FP & Missed GCs'); axes[2].set_xlabel('g–z colour')
    axes[2].legend()

    for ax in axes: ax.set_ylabel('Density')
    plt.suptitle('g–z Colour Distribution Recovery', fontsize=13, fontweight='bold')
    plt.tight_layout()

    send_telegram_photo(token, chat_id, fig,
        caption=f"✅ g–z colour recovery\n"
                f"FP: {fp_mask.sum()}  |  Missed GCs: {fn_mask.sum()}")
    plt.show()


## 11 · Explainability Suite

Explainability serves two distinct goals in this project:

1. **Model improvement** — understanding what the model attends to can reveal failure modes, guide architecture choices, and validate that learning is physically meaningful rather than spurious
2. **Scientific insight** — domain experts can use pixel-level attributions as a new observational tool: *which spatial and spectral features of a GC image drive the classification?*

We use four complementary techniques, each answering a different question.

---

### 11.1 · Integrated Gradients on the GP Mean

**Question: which pixels drove the classification decision?**

Integrated Gradients (Sundararajan et al. 2017) is a gradient-based attribution method that satisfies two key axioms — *sensitivity* (a feature that changes the prediction gets non-zero attribution) and *implementation invariance* (two functionally equivalent models yield the same attributions).

The attribution for pixel $i$ is:

$$\text{IG}_i(\mathbf{x}) = (x_i - b_i) \int_{\alpha=0}^{1} \frac{\partial F\!\left(\mathbf{b} + \alpha(\mathbf{x}-\mathbf{b})\right)}{\partial x_i}\, d\alpha$$

where $F$ is the model's predicted $P(\text{GC})$ and $\mathbf{b}$ is a **baseline** — a reference image representing "no signal". We use the **mean image of non-GC training examples** as the baseline. This is astronomically meaningful: the model must explain why this source differs from a typical non-GC background, rather than from an uninformative black image.

The integral is approximated with 50 steps of the Riemann sum along the straight-line path $\mathbf{b} \to \mathbf{x}$.

**Connection to the paper:** Dold & Fahrion used LIME on tabular features and identified g−z colour, magnitudes, and concentration indices as the most important variables. IG on our 2-channel images allows us to verify whether the model is attending to the *same physical regions* — the central brightness peak (a proxy for concentration index) and the relative flux in g vs z (a proxy for colour). If IG maps are centrally concentrated and asymmetric between channels, the model is learning the right physics.

In [ ]:
with notify_on_error(token, chat_id, "IG + baseline definition"):
    def integrated_gradients(model: DeepCGP,
                             img_tensor: torch.Tensor,    # (1, 2, H, W)
                             baseline: torch.Tensor,       # (1, 2, H, W)
                             target_class: int = 1,
                             steps: int = 50) -> torch.Tensor:
        """
        Compute Integrated Gradients for `target_class` (default: GC=1).
        Returns attribution map of shape (2, H, W).
        """
        model.eval()
        alphas = torch.linspace(0, 1, steps, device=img_tensor.device)
        grads  = []
    
        for alpha in alphas:
            interp = (baseline + alpha * (img_tensor - baseline)).detach().requires_grad_(True)
            mean, _ = model(interp)
            prob = F.softmax(mean, dim=-1)[0, target_class]
            model.zero_grad()
            prob.backward()
            grads.append(interp.grad.detach().clone())            # (1, 2, H, W)
    
        avg_grad = torch.stack(grads, 0).mean(0)                  # (1, 2, H, W)
        ig = ((img_tensor - baseline) * avg_grad).squeeze(0)      # (2, H, W)
        return ig
    
    
    # ── compute non-GC mean image as baseline ─────────────────────────────────────
    train_ds = FrameDataset(train_df)
    ngc_imgs = np.stack([train_ds[i][0].numpy()
                         for i in range(len(train_ds))
                         if train_ds.labels[i] == 0])[:2000]
    baseline_np = ngc_imgs.mean(axis=0)                           # (2, 20, 20)
    baseline_t  = torch.tensor(baseline_np).unsqueeze(0).to(device)
    
    print(f"Baseline computed from {len(ngc_imgs)} non-GC training images.")


In [ ]:
with notify_on_error(token, chat_id, "Integrated Gradients visualisation"):
    def pick_samples(y_true, y_pred, y_proba, n=4):
        tp_idx = np.where((y_true==1) & (y_pred==1))[0]
        fp_idx = np.where((y_true==0) & (y_pred==1))[0]
        fn_idx = np.where((y_true==1) & (y_pred==0))[0]
        tp_top = tp_idx[np.argsort(y_proba[tp_idx,1])[-n:]]
        fp_top = fp_idx[np.argsort(y_proba[fp_idx,1])[-n:]]
        fn_top = fn_idx[np.argsort(y_proba[fn_idx,1])[:n]]
        return tp_top, fp_top, fn_top

    tp_idx, fp_idx, fn_idx = pick_samples(y_true, y_pred, y_proba, n=4)
    test_ds = FrameDataset(test_df)

    def show_ig_panel(title, indices, label_str, n_cols=4):
        fig, axes = plt.subplots(3, n_cols, figsize=(4*n_cols, 10))
        fig.suptitle(title, fontsize=13, fontweight='bold')
        for col, idx in enumerate(indices):
            img_t, lbl = test_ds[idx]
            img_t = img_t.unsqueeze(0).to(device)
            ig     = integrated_gradients(best_model, img_t, baseline_t).cpu().numpy()
            img_np = img_t.squeeze(0).cpu().numpy()

            im0 = axes[0, col].imshow(img_np[0], cmap='Reds')
            axes[0, col].set_title(f"g-band | {label_str}")
            plt.colorbar(im0, ax=axes[0, col])

            im1 = axes[1, col].imshow(img_np[1], cmap='Greens')
            axes[1, col].set_title("z-band")
            plt.colorbar(im1, ax=axes[1, col])

            ig_abs = np.abs(ig).sum(axis=0)
            axes[2, col].imshow(img_np[0], cmap='Reds', alpha=0.5)
            axes[2, col].imshow(ig_abs, cmap='hot', alpha=0.6)
            axes[2, col].set_title("IG attribution")

            for row in range(3):
                axes[row, col].axis('off')

        plt.tight_layout()
        return fig

    fig_tp = show_ig_panel("True Positives — Integrated Gradients", tp_idx, "TP")
    send_telegram_photo(token, chat_id, fig_tp,
        caption="✅ IG — True Positives (top P(GC) correct predictions)")
    plt.show(); plt.close(fig_tp)

    fig_fp = show_ig_panel("False Positives — Integrated Gradients", fp_idx, "FP")
    send_telegram_photo(token, chat_id, fig_fp,
        caption="✅ IG — False Positives (non-GCs misclassified as GC)")
    plt.show(); plt.close(fig_fp)

    fig_fn = show_ig_panel("Missed GCs (False Negatives) — IG", fn_idx, "FN")
    send_telegram_photo(token, chat_id, fig_fn,
        caption="✅ IG — False Negatives (GCs the model missed)")
    plt.show(); plt.close(fig_fn)


### 11.2 · XUE — Uncertainty Attribution

**Question: which pixels make the model uncertain?**

This technique is unique to probabilistic models and has no CNN equivalent. We compute the gradient of the **predictive variance** with respect to the input:

$$\text{XUE}_i(\mathbf{x}) = \left|\frac{\partial \text{Var}[f^*(\mathbf{x})]}{\partial x_i}\right|$$

High XUE on a pixel means that small changes to that pixel strongly affect how uncertain the model is — regardless of which class it predicts. This is a fundamentally different signal from IG:

- **High IG, low XUE** → the model is both confident and attending to this pixel for its decision
- **High IG, high XUE** → the model uses this pixel but is unsure about it
- **Low IG, high XUE** → the pixel doesn't drive the classification but destabilises the model's confidence

In practice, sources with high overall XUE are **near the decision boundary** or **atypical** — these are exactly the candidates most valuable to flag for follow-up spectroscopic observation, since they represent potential GC discoveries the model is uncertain about.

We compute XUE separately for each band (g and z), which allows comparison of how much each frequency contributes to the model's uncertainty.

In [ ]:
with notify_on_error(token, chat_id, "XUE — uncertainty attribution"):
    def xue(model, img_tensor):
        """|∂Var(f*)/∂x| per channel. Returns (2, H, W)."""
        model.eval()
        x = img_tensor.detach().clone().requires_grad_(True)
        _, var = model(x)
        model.zero_grad()
        var.sum().backward()
        return x.grad.abs().squeeze(0)

    def show_xue_panel(title, indices):
        n = len(indices)
        fig, axes = plt.subplots(2, n, figsize=(4*n, 7))
        fig.suptitle(title, fontsize=12, fontweight='bold')
        for col, idx in enumerate(indices):
            img_t, lbl = test_ds[idx]
            attr = xue(best_model, img_t.unsqueeze(0).to(device)).cpu().numpy()
            for ch, cmap, band in zip([0,1], ['Reds','hot'],
                                       ['g-band XUE','z-band XUE']):
                axes[ch, col].imshow(attr[ch], cmap=cmap)
                axes[ch, col].set_title(f"{band} | idx={idx}")
                axes[ch, col].axis('off')
        plt.tight_layout()
        return fig

    fig_xue_tp = show_xue_panel("XUE — True Positives", tp_idx)
    send_telegram_photo(token, chat_id, fig_xue_tp,
        caption="✅ XUE — uncertainty attribution on True Positives")
    plt.show(); plt.close(fig_xue_tp)

    fig_xue_fp = show_xue_panel("XUE — False Positives", fp_idx)
    send_telegram_photo(token, chat_id, fig_xue_fp,
        caption="✅ XUE — uncertainty attribution on False Positives")
    plt.show(); plt.close(fig_xue_fp)


### 11.3 · Inducing Point Visualisation — Learned Patch Prototypes

**Question: what image patches does the model consider prototypical?**

This is an interpretability technique with **no equivalent in the paper's CNN**. The inducing inputs $\mathbf{Z} \in \mathbb{R}^{M \times P}$ of each GP layer are $M$ learned patch vectors that serve as a "comparison dictionary": the GP posterior at any new patch is computed via its similarity (through the kernel) to each inducing point.

After training, $\mathbf{Z}$ can be reshaped to $(M, C_\text{in}, k, k)$ and visualised directly as image patches. We rank them by their **mean activation on true GC test images**:

$$\text{activation}_m = \mathbb{E}_{\mathbf{x} \in \text{GC test}}\!\left[\left|\mathbf{A}_m\right|\right], \qquad A_{im} = \frac{K(\mathbf{p}_i, \mathbf{z}_m)}{[K_{\mathbf{ZZ}}]_{mm}}$$

The top-ranked patches are the prototypes the model most heavily uses when recognising GCs. Domain astronomers can inspect these directly to verify they correspond to known morphological features: a centrally concentrated point source with the right colour ratio.

This connects to the concept of **prototype networks** in interpretable ML, but emerges naturally from the GP framework without any additional design choices.

In [ ]:
with notify_on_error(token, chat_id, "Inducing point visualisation"):
    def get_top_inducing_patches(model, layer_idx=0, test_loader=None, top_k=16):
        layer = model.layers[layer_idx]
        k, pad, stride = layer.k, layer.padding, layer.stride
        model.eval()
        gc_activations = []
        with torch.no_grad():
            for x, y in test_loader:
                x = x.to(device)
                gc_mask = (y == 1)
                if gc_mask.sum() == 0: continue
                x_gc = x[gc_mask]
                patches = F.unfold(x_gc, kernel_size=k, padding=pad, stride=stride)
                patches = patches.transpose(1,2)
                L_z  = layer._chol_kzz()
                K_xz = layer._rbf(patches,
                                   layer.Z.unsqueeze(0).expand(x_gc.shape[0],-1,-1))
                A = torch.cholesky_solve(K_xz.transpose(-1,-2), L_z).transpose(-1,-2)
                gc_activations.append(A.abs().mean(dim=[0,1]).cpu().numpy())

        mean_act = np.stack(gc_activations, 0).mean(0)
        top_idx  = np.argsort(mean_act)[::-1][:top_k]
        Z_np = layer.Z.detach().cpu().numpy()
        in_ch = layer.P // (k * k)
        patches_out = Z_np[top_idx].reshape(-1, in_ch, k, k)
        return patches_out, mean_act[top_idx]

    top_patches_ext, activations_ext = get_top_inducing_patches(
        best_model, layer_idx=0, test_loader=test_dl, top_k=16)

    fig, axes = plt.subplots(4, 4, figsize=(10, 10))
    fig.suptitle("Top-16 Inducing Points — Extractor Layer\n"
                 "(ranked by mean activation on true GC test images)", fontsize=12)
    for i, ax in enumerate(axes.flat):
        patch = top_patches_ext[i]
        h, w  = patch.shape[1], patch.shape[2]
        rgb   = np.zeros((h, w, 3))
        p0 = patch[0]; p0 = (p0 - p0.min()) / (p0.max() - p0.min() + 1e-8)
        p1 = patch[1]; p1 = (p1 - p1.min()) / (p1.max() - p1.min() + 1e-8)
        rgb[:,:,0] = p0
        rgb[:,:,1] = p1
        ax.imshow(rgb, interpolation='nearest')
        ax.set_title(f"act={activations_ext[i]:.3f}", fontsize=8)
        ax.axis('off')
    plt.tight_layout()

    send_telegram_photo(token, chat_id, fig,
        caption="✅ Top-16 inducing point patches (red=g-band, green=z-band)\n"
                "These are the GP's learned 'GC prototype patches'.")
    plt.show()


### 11.4 · Per-Channel Attribution Ratio (g vs z)

**Question: is the model using colour, or just brightness?**

Since our images have exactly two channels corresponding to two photometric bands, we can compute what fraction of the total IG attribution comes from each band:

$$r(\mathbf{x}) = \frac{\|\text{IG}_z(\mathbf{x})\|_1}{\|\text{IG}_g(\mathbf{x})\|_1 + \|\text{IG}_z(\mathbf{x})\|_1} \in [0, 1]$$

A value near 1 means the model's decision was dominated by the z-band; near 0 means it was dominated by the g-band; near 0.5 means both contributed equally.

**Physical interpretation:**
- A model that learned colour (g−z) will show $r$ values that **correlate with the actual g−z colour** of the source — redder GCs (higher g−z) should show higher z-band attribution
- A model that learned only brightness/concentration will show $r \approx 0.5$ regardless of colour, since both bands are equally informative about a point source's brightness peak

We visualise $r$ as a function of: the actual g−z colour (to check the correlation), the predicted $P(\text{GC})$ (to check whether confident predictions rely more on colour), and separately for GCs vs non-GCs (to check whether the model uses different features for different classes).

This provides a direct quantitative connection between our image-based model and the tabular feature importance results from the paper.

In [ ]:
with notify_on_error(token, chat_id, "Per-channel (g vs z) attribution ratio"):
    def batch_channel_ratio(model, loader, baseline_t, steps=20):
        model.eval()
        ratios, labels, probs = [], [], []
        for x, y in loader:
            x = x.to(device)
            for i in range(len(y)):
                ig    = integrated_gradients(model, x[i:i+1], baseline_t,
                                             steps=steps).cpu().numpy()
                ig_g  = np.abs(ig[0]).sum()
                ig_z  = np.abs(ig[1]).sum()
                ratios.append(ig_z / (ig_g + ig_z + 1e-8))
                labels.append(y[i].item())
            with torch.no_grad():
                mean, var = model(x)
                std = var.clamp(min=1e-6).sqrt()
                p   = F.softmax(Normal(mean, std).rsample((100,)),
                                dim=-1).mean(0)[:,1].cpu().numpy()
                probs.extend(p.tolist())
        return np.array(ratios), np.array(labels), np.array(probs)

    sub_test = test_df.sample(n=min(300, len(test_df)), random_state=42)
    sub_dl   = make_loader(sub_test, batch_size=32, shuffle=False, num_workers=0)
    ratios, sub_labels, sub_probs = batch_channel_ratio(best_model, sub_dl, baseline_t)
    sub_colours = sub_test['colour'].values

    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].hist(ratios[sub_labels==1], bins=30, alpha=0.7, color='tomato',
                 label='GC',     density=True)
    axes[0].hist(ratios[sub_labels==0], bins=30, alpha=0.7, color='steelblue',
                 label='non-GC', density=True)
    axes[0].set_xlabel('z-band attribution fraction  |IG_z| / (|IG_g|+|IG_z|)')
    axes[0].set_ylabel('Density'); axes[0].legend()
    axes[0].set_title('Channel Attribution by Class')

    gc_mask_sub = sub_labels == 1
    sc = axes[1].scatter(sub_colours[gc_mask_sub], ratios[gc_mask_sub],
                         c=sub_probs[gc_mask_sub], cmap='viridis', alpha=0.6, s=20)
    plt.colorbar(sc, ax=axes[1], label='P(GC)')
    axes[1].set_xlabel('g–z colour'); axes[1].set_ylabel('z-band attribution fraction')
    axes[1].set_title('GCs: Attribution Ratio vs Actual Colour')

    axes[2].scatter(sub_probs, ratios, c=sub_labels, cmap='RdBu', alpha=0.4, s=15)
    axes[2].set_xlabel('P(GC)'); axes[2].set_ylabel('z-band attribution fraction')
    axes[2].set_title('Attribution Ratio vs Confidence')

    plt.suptitle('Per-Channel (g vs z) Attribution Analysis',
                 fontsize=13, fontweight='bold')
    plt.tight_layout()

    send_telegram_photo(token, chat_id, fig,
        caption="✅ Per-channel attribution (g vs z)\n"
                "A high z-fraction means the model responds to colour (g–z),\n"
                "matching LIME findings from the paper.")
    plt.show()


## 12 · Cross-Cluster Transfer Analysis (Virgo ↔ Fornax)

### Physical motivation

Virgo (~16.5 Mpc) and Fornax (~20 Mpc) are the two nearest large galaxy clusters and are the source of both our training catalogues. Their GC populations are physically similar — same stellar age, same metallicity distribution, same bimodal colour — but at different distances they subtend slightly different angular sizes on the sky. A model that truly learned the physics of GCs rather than distance-specific appearance should transfer well across clusters.

Dold & Fahrion report TPR ≈ 90–94 % even in the cross-cluster direction, which validates the physical generalisability of image-based GC selection.

### The DCGP's unique advantage

A CNN produces a scalar confidence score and has no principled way to express "I've never seen anything like this". Our DCGP does: the **predictive variance inflates** for test inputs whose patches are dissimilar to the training inducing points. When the model is applied to a cluster it was not trained on, we expect:

$$\text{Var}[f^*(\mathbf{x}_\text{Fornax})] > \text{Var}[f^*(\mathbf{x}_\text{Virgo})] \quad \text{(when trained on Virgo)}$$

This **OOD uncertainty inflation** can be measured and used as a quality filter: by restricting predictions to low-variance sources, we expect to recover a high-TPR sub-catalogue even on the out-of-distribution cluster. We plot the TPR as a function of the fraction of sources retained after applying a variance threshold — a curve that a CNN simply cannot produce.

In [ ]:
with notify_on_error(token, chat_id, "Cross-cluster evaluation"):
    def cross_cluster_eval(train_df, test_df, tag, n_epochs=30):
        print(f"\n{'='*55}\n {tag}\n{'='*55}")
        cnts = train_df['y'].value_counts()
        n0, n1 = int(cnts.get(False, 1)), int(cnts.get(True, 1))
        tot = n0 + n1
        cw  = torch.tensor([tot/(2*n0), tot/(2*n1)],
                           dtype=torch.float32).to(device)
        print(f"  Train: {len(train_df):,} | Test: {len(test_df):,}")

        tr_dl = make_loader(train_df, shuffle=True,  num_workers=0)
        te_dl = make_loader(test_df,  shuffle=False, num_workers=0)
        model = build_model(best_params, device)
        model.initialize_inducing_points(tr_dl)
        opt   = torch.optim.Adam(model.parameters(), lr=best_params['learning_rate'])

        for epoch in range(n_epochs):
            train_epoch(model, tr_dl, opt, epoch, cw,
                        warmup_epochs=5, dataset_size=len(train_df))

        y_t, y_p, y_pred_, y_std_ = predict_dataset(model, te_dl)
        tn_c, fp_c, fn_c, tp_c = confusion_matrix(y_t, y_pred_).ravel()
        tpr_cc = tp_c / (tp_c + fn_c)
        fdr_cc = fp_c / (fp_c + tp_c)
        auc_cc = roc_auc_score(y_t, y_p[:,1])
        print(f"  TPR: {tpr_cc:.3f}  FDR: {fdr_cc:.3f}  AUC: {auc_cc:.3f}")

        send_telegram_msg(token, chat_id,
            f"✅ *Cross-cluster* — {tag}\n"
            f"TPR: `{tpr_cc:.3f}`  FDR: `{fdr_cc:.3f}`  AUC: `{auc_cc:.3f}`")

        gc_stds = y_std_[y_t == 1, 1]
        ng_stds = y_std_[y_t == 0, 1]

        thresholds   = np.percentile(y_std_[:,1], np.arange(10, 100, 10))
        tpr_filt, frac_kept = [], []
        for thr in thresholds:
            mask = y_std_[:,1] <= thr
            if mask.sum() == 0 or (y_t[mask] == 1).sum() == 0: continue
            tp_f = ((y_pred_[mask]==1) & (y_t[mask]==1)).sum()
            fn_f = ((y_pred_[mask]==0) & (y_t[mask]==1)).sum()
            tpr_filt.append(tp_f / (tp_f + fn_f))
            frac_kept.append(mask.mean())

        return {'tpr': tpr_cc, 'fdr': fdr_cc, 'auc': auc_cc,
                'gc_stds': gc_stds, 'ng_stds': ng_stds,
                'tpr_filt': tpr_filt, 'frac_kept': frac_kept, 'tag': tag}

    results_vcc_to_fcc = cross_cluster_eval(
        vcc_df, fcc_df, "Train: Virgo (VCC) → Test: Fornax (FCC)")
    results_fcc_to_vcc = cross_cluster_eval(
        fcc_df, vcc_df, "Train: Fornax (FCC) → Test: Virgo (VCC)")


In [ ]:
with notify_on_error(token, chat_id, "Cross-cluster visualisation"):
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))

    tags = ['VCC→FCC', 'FCC→VCC']
    tprs = [results_vcc_to_fcc['tpr'], results_fcc_to_vcc['tpr']]
    fdrs = [results_vcc_to_fcc['fdr'], results_fcc_to_vcc['fdr']]
    x = np.arange(len(tags)); w = 0.35
    axes[0].bar(x - w/2, tprs, w, label='TPR', color='tomato')
    axes[0].bar(x + w/2, fdrs, w, label='FDR', color='steelblue')
    axes[0].axhline(0.90, color='grey', ls='--', label='Paper best TPR')
    axes[0].axhline(0.08, color='navy', ls=':', label='Paper best FDR')
    axes[0].set_xticks(x); axes[0].set_xticklabels(tags)
    axes[0].set_ylim(0, 1.05); axes[0].legend()
    axes[0].set_title('Cross-Cluster TPR / FDR')

    for res, col, lbl in [(results_vcc_to_fcc, 'tomato', 'VCC→FCC'),
                           (results_fcc_to_vcc, 'steelblue', 'FCC→VCC')]:
        axes[1].hist(res['gc_stds'], bins=40, alpha=0.5, color=col,
                     label=f'{lbl} GC std', density=True)
    axes[1].set_xlabel('Predictive std on GC class'); axes[1].set_ylabel('Density')
    axes[1].set_title('Uncertainty on True GCs (OOD)'); axes[1].legend()

    for res, col, lbl in [(results_vcc_to_fcc, 'tomato', 'VCC→FCC'),
                           (results_fcc_to_vcc, 'steelblue', 'FCC→VCC')]:
        axes[2].plot(res['frac_kept'], res['tpr_filt'], 'o-', color=col, label=lbl)
    axes[2].set_xlabel('Fraction kept (low uncertainty)'); axes[2].set_ylabel('TPR')
    axes[2].set_title('TPR vs Uncertainty Filter'); axes[2].legend()

    plt.suptitle('Cross-Cluster Transfer Analysis', fontsize=13, fontweight='bold')
    plt.tight_layout()

    send_telegram_photo(token, chat_id, fig,
        caption=f"✅ Cross-cluster transfer\n"
                f"VCC→FCC: TPR={results_vcc_to_fcc['tpr']:.3f} FDR={results_vcc_to_fcc['fdr']:.3f}\n"
                f"FCC→VCC: TPR={results_fcc_to_vcc['tpr']:.3f} FDR={results_fcc_to_vcc['fdr']:.3f}")
    plt.show()


## 13 · Ablation Study

To understand the contribution of each design choice, we train six model variants on identical data and evaluate on the same held-out test set. Each variant changes exactly one thing from the full model.

| Variant | Change | What it tests |
|---------|--------|---------------|
| **Full model** | — | Baseline |
| **A1 – No class weighting** | Uniform ELBO weights | Does weighted loss help with the 1:3.4 imbalance? |
| **A2 – 1 GP layer** | Remove ExtractorConvGP | Is the two-layer depth necessary? |
| **A3 – M = 64** | Fewer inducing points | How much does the inducing set size matter? |
| **A4 – kernel = 1** | 1×1 "patches" (no spatial context) | Does the spatial receptive field matter? |
| **A5 – No KL (β = 0)** | Pure maximum likelihood | What does the Bayesian regularisation contribute? |

A4 is particularly informative: if a kernel-1 model performs comparably, the model is essentially doing pixel-wise classification without using neighbourhood structure — which would be surprising given that concentration (the spatial profile of brightness) is a key GC feature. A5 tests whether the GP regularisation is actually doing useful work or whether the model is learning a degenerate posterior.

In [ ]:
with notify_on_error(token, chat_id, "Ablation study — running variants"):
    def ablation_run(name, override_params, no_kl=False, n_layers=2, n_epochs=30):
        print(f"  Running: {name}...")
        send_telegram_msg(token, chat_id, f"🔬 Ablation: *{name}*...")
        params = {**best_params, **override_params}
        M, ks, oc = params['M'], params['kernel_size'], params['out_channels']
        s   = np.exp(params.get('log_s',  best_params['log_s']))
        ls_ = np.exp(params.get('log_ls', best_params['log_ls']))
        pad = ks // 2

        if n_layers == 1:
            classifier = ClassifierConvGP(device, 2, (20,20), 2, ks, M, pad, 1, s, ls_, 'cls')
            model = DeepCGP([classifier]).to(device)
        else:
            h_out = (20 + 2*pad - ks) + 1
            extractor  = ExtractorConvGP(device, 2, oc, ks, M, pad, 1, s, ls_, 'ext')
            classifier = ClassifierConvGP(device, oc, (h_out, h_out), 2, ks, M, pad, 1, s, ls_, 'cls')
            model = DeepCGP([extractor, classifier]).to(device)

        model.initialize_inducing_points(train_dl)
        opt = torch.optim.Adam(model.parameters(), lr=params['learning_rate'])
        cw  = CLASS_WEIGHTS if not override_params.get('uniform_weights') \
              else torch.ones(2, device=device)

        for epoch in range(n_epochs):
            model.train()
            eff_beta = 0.0 if no_kl else min(1.0, (epoch+1)/10)
            for x, y in train_dl:
                x, y = x.to(device), y.to(device)
                opt.zero_grad()
                loss, _ = elbo_loss(model, x, y, cw, num_samples=5,
                                    beta=eff_beta, dataset_size=len(train_df))
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 5.0)
                opt.step()

        y_t, y_p, y_pred_, _ = predict_dataset(model, test_dl, n_mc=200)
        tn_, fp_, fn_, tp_ = confusion_matrix(y_t, y_pred_).ravel()
        result = {
            'name'    : name,
            'TPR'     : tp_/(tp_+fn_),
            'FDR'     : fp_/(fp_+tp_),
            'Accuracy': (tp_+tn_)/len(y_t),
            'AUC'     : roc_auc_score(y_t, y_p[:,1])
        }
        print(f"    TPR={result['TPR']:.3f}  FDR={result['FDR']:.3f}  "
              f"Acc={result['Accuracy']:.3f}  AUC={result['AUC']:.3f}")
        return result

    ablation_results = []
    ablation_results.append(ablation_run("Full model (baseline)",           {}))
    ablation_results.append(ablation_run("A1 – No class weighting",         {'uniform_weights': True}))
    ablation_results.append(ablation_run("A2 – 1 GP layer",                 {}, n_layers=1))
    ablation_results.append(ablation_run("A3 – M=64 (fewer inducing pts)",  {'M': 64}))
    ablation_results.append(ablation_run("A4 – kernel=1 (no spatial ctx)",  {'kernel_size': 1}))
    ablation_results.append(ablation_run("A5 – No KL (β=0, pure MLE)",      {}, no_kl=True))


In [ ]:
with notify_on_error(token, chat_id, "Ablation study — results & plot"):
    abl_df = pd.DataFrame(ablation_results).set_index('name')
    print("\n── Ablation Results ──────────────────────────────────────")
    print(abl_df.round(3).to_string())

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    for ax, (met, col) in zip(axes, [('TPR','tomato'), ('FDR','steelblue'), ('AUC','seagreen')]):
        bars = ax.barh(abl_df.index, abl_df[met], color=col, alpha=0.8)
        ax.set_title(met)
        xlim = (0, 1) if met in ('TPR','FDR') else (0.5, 1)
        ax.set_xlim(*xlim)
        ax.axvline(abl_df[met].iloc[0], ls='--', color='black', lw=1, label='Full model')
        for bar, val in zip(bars, abl_df[met]):
            ax.text(val + 0.005, bar.get_y() + bar.get_height()/2,
                    f'{val:.3f}', va='center', fontsize=8)
        ax.legend()

    plt.suptitle('Ablation Study Results', fontsize=13, fontweight='bold')
    plt.tight_layout()

    # Build a compact table string for the caption
    table_lines = ["Ablation summary:"]
    for row in abl_df.itertuples():
        table_lines.append(
            f"{row.Index[:28]:<28} TPR={row.TPR:.3f} FDR={row.FDR:.3f} AUC={row.AUC:.3f}")
    caption = "\n".join(table_lines)

    send_telegram_photo(token, chat_id, fig, caption=caption)
    plt.show()

    send_telegram_msg(token, chat_id,
        "🎉 *Notebook complete!* All sections finished successfully.")


---

## Summary

| Component | Design choice | Scientific justification |
|-----------|--------------|--------------------------|
| **DCGP over CNN** | Probabilistic model with posterior variance | Calibrated uncertainty enables OOD detection and per-source quality flags |
| **Weighted ELBO** | Inverse-frequency class weights in data-fit term | Preserves all ~85 k training examples; avoids discarding majority-class information |
| **KL warm-up** | β annealed 0 → 1 over warmup epochs | Prevents prior from collapsing posterior before data is fit |
| **k-means inducing init** | Cached per (M, P, layer, dataset) | Initialises inducing points in data-space; cached to avoid recomputing across Optuna trials |
| **Integrated Gradients** | Non-GC mean image as baseline | Pixel attribution of classification decision; validates model learns concentration + colour |
| **XUE** | Gradient of predictive variance | Identifies uncertain sources for follow-up; unique to probabilistic models |
| **Inducing point viz** | Rank Z by GC activation | Learned prototype patches; connects GP internals to physical GC morphology |
| **g/z attribution ratio** | Per-channel IG mass | Directly tests whether model uses colour (g−z), matching tabular LIME results from the paper |
| **Cross-cluster transfer** | Train VCC → test FCC and vice versa | Tests physical generalisability; GP uncertainty inflation on OOD data is novel vs CNN |
| **Ablation study** | 6 controlled variants | Quantifies contribution of each design choice |